In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np
import pandas as pd

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Katalog roboczy (notatnik): {notebook_dir}")
print(f"Katalog główny projektu: {project_root}")
print(f"Załadowano ścieżkę modułów: {src_path}")

In [ ]:
from hep_tracking.dataset import TrackDataset
from hep_tracking.config import KNNModelConfig
from hep_tracking.models import BaseKNN, ScipyCKDTree, SklearnKNN
from hep_tracking.benchmark import ANNBenchmarkRunner
from hep_tracking.plots import plot_3d_hits, plot_distance_distributions, plot_scaling

print("Zależności modułu hep_tracking załadowane pomyślnie!")

In [ ]:
%matplotlib inline

data_dir = project_root / "data"
sanity_dataset_path = data_dir / "dataset_hard_10k.npz"

if sanity_dataset_path.exists():
    dataset_sanity = TrackDataset.from_npz(sanity_dataset_path)
    
    print(f"Wczytano zbiór kontrolny: {sanity_dataset_path.name}")
    print(f"Kształt macierzy cech: {dataset_sanity.X.shape}")
    
    plot_3d_hits(dataset_sanity)
    plot_distance_distributions(dataset_sanity)
else:
    print(f"Nie znaleziono pliku {sanity_dataset_path.name}. Upewnij się, że wygenerowałeś dane (make generate).")

In [ ]:
class NumpyBruteForce(BaseKNN):
    def fit(self, X: np.ndarray) -> None:
        self.X_train = X

    def kneighbors(self, X: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
        n_samples = X.shape[0]
        squared_norms = np.sum(X * X, axis=1)
        
        distances = (squared_norms[:, None] + squared_norms[None, :] - 2.0 * (X @ self.X_train.T))
        distances = np.maximum(distances, 0.0)
        np.fill_diagonal(distances, np.inf)
        
        indices = np.argpartition(distances, k, axis=1)[:, :k]
        
        nearest_distances = np.empty((n_samples, k), dtype=np.float32)
        nearest_indices = np.empty((n_samples, k), dtype=np.int64)
        
        for i in range(n_samples):
            order = np.argsort(distances[i, indices[i]])
            nearest_indices[i] = indices[i, order]
            nearest_distances[i] = np.sqrt(distances[i, indices[i, order]])
            
        return nearest_distances, nearest_indices

models_configs = [
    KNNModelConfig("Numpy Brute Force", NumpyBruteForce, {}),
    KNNModelConfig("Scipy cKDTree", ScipyCKDTree, {"workers": -1}),
    KNNModelConfig("Sklearn KDTree", SklearnKNN, {"algorithm": "kd_tree"}),
    KNNModelConfig("Sklearn BallTree", SklearnKNN, {"algorithm": "ball_tree"})
]

try:
    import cupy as cp
    class CuPyBruteForce(BaseKNN):
        def fit(self, X: np.ndarray) -> None:
            self.X_train = cp.asarray(X)

        def kneighbors(self, X: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
            X_gpu = cp.asarray(X)
            n_samples = X_gpu.shape[0]
            squared_norms = cp.sum(X_gpu * X_gpu, axis=1)
            
            distances = (squared_norms[:, None] + squared_norms[None, :] - 2.0 * (X_gpu @ self.X_train.T))
            distances = cp.maximum(distances, 0.0)
            cp.fill_diagonal(distances, cp.inf)
            
            indices = cp.argpartition(distances, k, axis=1)[:, :k]
            
            dist_cpu, idx_cpu = distances.get(), indices.get()
            nearest_distances = np.empty((n_samples, k), dtype=np.float32)
            nearest_indices = np.empty((n_samples, k), dtype=np.int64)
            
            for i in range(n_samples):
                order = np.argsort(dist_cpu[i, idx_cpu[i]])
                nearest_indices[i] = idx_cpu[i, order]
                nearest_distances[i] = np.sqrt(dist_cpu[i, idx_cpu[i, order]])
                
            return nearest_distances, nearest_indices
            
    models_configs.insert(1, KNNModelConfig("CuPy Brute Force", CuPyBruteForce, {}))
except ImportError:
    print("Moduł 'cupy' nie został znaleziony. Pomijam model 'CuPy Brute Force'.")

target_sizes = {"1k": 1000, "10k": 10000, "100k": 100000, "1M": 1000000}
dataset_modes = ["easy", "hard"]
k_neighbors = 5

runner = ANNBenchmarkRunner(k_neighbors=k_neighbors, warmup_runs=1, num_runs=3)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

all_results = []

for mode in dataset_modes:
    print(f"\n{'='*45}")
    print(f" TRYB DANYCH: {mode.upper()}")
    print(f"{'='*45}")

    for size_label, size_val in target_sizes.items():
        filename = data_dir / f"dataset_{mode}_{size_label}.npz"
        
        if not filename.exists():
            print(f"\n[BRAK PLIKU] {filename.name} - Pomijam.")
            continue

        dataset = TrackDataset.from_npz(filename)
        print(f"\n Załadowano: {size_label} ({len(dataset)} hitów)")
        dataset_name = f"{mode}_{size_label}"

        # Wyznaczenie punktu odniesienia dla recallu (chociaż w dokładnym kNN wyniesie on zawsze 1.0)
        exact_knn = ScipyCKDTree(workers=-1)
        exact_knn.fit(dataset.X)
        _, true_idx = exact_knn.kneighbors(dataset.X, k=k_neighbors)

        # Filtrowanie zbyt wolnych modeli dla N = 1M
        active_configs = models_configs
        if size_val > 100000:
            active_configs = [
                c for c in models_configs 
                if c.name not in ["Numpy Brute Force", "CuPy Brute Force", "Sklearn BallTree"]
            ]
            print("  > Pomijam modele Brute Force oraz BallTree (Zbyt wolne dla 1M)")

        df_size = runner.run(
            models_configs=active_configs, 
            datasets={dataset_name: dataset}, 
            ground_truth={dataset_name: true_idx}
        )
        
        df_size['Mode'] = mode
        df_size['Size_Label'] = size_label
        df_size['Size'] = size_val
        all_results.append(df_size)

print("\nBenchmark zakończony sukcesem!")
results_df = pd.concat(all_results, ignore_index=True)
display(results_df.head())

In [ ]:
for mode in dataset_modes:
    df_mode = results_df[results_df['Mode'] == mode]
    
    plot_scaling(
        df=df_mode,
        x_col="Size",
        y_col="Query_Time_s",
        title=f"Skalowalność kNN - tryb: {mode.upper()}",
        log_x=True,
        log_y=True
    )